# Fine-tuning a pretrained ResNet18 (P >> N)

Genuine dataset-overparam: ImageNet-pretrained ResNet18 (~11M params) fine-tuned on small CIFAR subsets. Test accuracy vs N (P/N), Sven vs AdamW/SGD/Muon.

> Loads the fresh Gram-backend results. Robust to partial data (plots whatever has finished).

In [1]:
import sys, json
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
sys.path.insert(0, '.')
from style import load_results, average_over_seeds, set_style
from analysis_helpers import (add_derived, best_per_method, seed_mean_best, config_runs, seed_band,
                              errorbar_seeds, fmt_pm, epochs_to_target_table, valid, loss_curve, method_order,
                              n_params_of)
from style import DATASET_TITLES, metric_label
from style import clipped_yerr
from pathlib import Path
set_style()
PLOT_DIR = Path('plots_v2/finetune'); PLOT_DIR.mkdir(parents=True, exist_ok=True)
from style import method_color
def sven_color(m): return method_color(m)   # global optimizer colours (style.METHOD_COLORS)
def sven_lw(m):    return 2.6 if m=='Sven' else 1.6

In [2]:
try:
    df = add_derived(load_results('exp_finetune_cifar_smallN'))
except FileNotFoundError as e:
    print('missing exp_finetune_cifar_smallN -- see RERUNS_NEEDED.md item 7; nothing to plot'); df = None
if df is not None:
    P = n_params_of(df, 11_181_642, 'ResNet18')   # from the records when they carry n_params


missing exp_finetune_cifar_smallN -- see RERUNS_NEEDED.md item 7; nothing to plot


### Best val accuracy vs training-set size (P/N)

In [3]:
if df is not None:
    # Best CONFIG per (optimizer, N) by seed-mean final val loss; error bars = seed spread.
    b=best_per_method(df, by='final_val_loss', extra_group=['n_data'])
    fig,ax=plt.subplots(figsize=(6.8,4.4))
    for m in method_order(b['method'].unique()):
        errorbar_seeds(ax, b[b.method==m], 'n_data', 'final_val_acc', color=sven_color(m), lw=sven_lw(m), label=m)
    ax.set_xscale('log'); ax.set_xlabel('N train (P/N = %.0f/N)'%P); ax.set_ylabel(metric_label('final_val_acc')); ax.legend(fontsize=8)
    plt.tight_layout(); plt.savefig(PLOT_DIR/'finetune_acc_vs_N.pdf',bbox_inches='tight'); plt.show()


### Table

In [4]:
if df is not None:
    b=best_per_method(df, by='final_val_loss', extra_group=['n_data'])
    for nd in sorted(b['n_data'].unique()):
        print(f'== N={nd}  (P/N={P/nd:.0f}) ==  (best config per optimizer; mean ± std over seeds)')
        for _,r in b[b.n_data==nd].sort_values('final_val_loss').iterrows():
            print(f"  {r.method:8s} val_acc={fmt_pm(r,'final_val_acc','.3f')}  val_loss={fmt_pm(r,'final_val_loss','.4f')}  seeds={r.n_seeds}")
